In [1]:
!nvidia-smi

Thu May 14 23:57:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
#import rouge_score to evaluate model
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.3/495.3 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 17.2 MB/s eta 0:00:00


In [3]:
#accelerate is commonly used to assign jobs in a better way to the GPUs
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 37.5 MB/s eta 0:00:00


##Purpose of accelerate:
Ease of Multi-Device Training: Whether you're using multiple GPUs or TPUs, accelerate makes it easier to scale your model across devices with minimal code changes.

Mixed Precision: It allows models to be trained using mixed precision, which can speed up training and reduce memory usage.

Zero Redundancy Optimizer (ZeRO): Helps manage large models by efficiently splitting the model across multiple devices.

Offload to CPU/SSD: Useful for large models that may not fit entirely into GPU memory, by allowing parts of the model or optimizer to be offloaded to CPU or even SSD.

In [7]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00


In [8]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
import pandas as pd
#Autotokenizer used to convert texts to tokens. AutoModelForSeq2SeqLM is used to load the huggingface model.

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

from datasets import load_dataset
!pip install evaluate # Install the missing 'evaluate' library
import evaluate # Changed from 'from datasets import load_metric'
import nltk
from nltk.tokenize import sent_tokenize

from tqdm import tqdm
import torch
nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

The code below is a demonstration of how the huggingface model is used for a sammple input data. Just shows the basic functionality.

In [10]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:

model_ckpt = "google/pegasus-cnn_dailymail"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)



In [ ]:
#dowload & unzip data

!wget https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip
!unzip summarizer-data.zip


### Fine tuning

In [11]:
#LEts first get the original model which we need to fin tune using training.
#In huggingface, the model we use for summarization, the same model is used for tokenization
model_ckpt = "google/pegasus-cnn_dailymail"
#Tokenizer to convert texts to tokens (tokenizer)
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
#Here we load the model, same model as tokenizer is used
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [12]:
#Download and unzip the data to be used with the model
#check about the Samsum dataset at https://huggingface.co/datasets/knkarthick/samsum
!wget https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip
!unzip summarizer-data.zip


--2026-05-15 00:48:04--  https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/entbappy/Branching-tutorial/master/summarizer-data.zip [following]
--2026-05-15 00:48:04--  https://raw.githubusercontent.com/entbappy/Branching-tutorial/master/summarizer-data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7903594 (7.5M) [application/zip]
Saving to: ‘summarizer-data.zip’

summarizer-data.zip 100%[===================>]   7.54M  --.-KB/s    in 0.02s   

2026-05-15 00:48:05 (483 MB/s) - ‘summarizer-data.zip’ saved [7903594/

In [13]:
#load_from_disk function of huggingface loads its contents in a dictionary format
dataset_samsum = load_from_disk('samsum_dataset')
dataset_samsum

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
})

In [14]:
# we are going to get some more information about the test, train and validation dataset
split_lengths = [len(dataset_samsum[split])for split in dataset_samsum]

print(f"Split lengths: {split_lengths}")
print(f"Features: {dataset_samsum['train'].column_names}")
print("\ndialogue")

print(dataset_samsum["test"][1]["dialogue"]) #dialogue column for the first record

print("\nsummary")

print(dataset_samsum["test"][1]["summary"]) #summary column for the first record

print("\ndialogue")

print(dataset_samsum["test"][2]["dialogue"]) #dialogue column for the 2nd record

print("\nsummary")

print(dataset_samsum["test"][2]["summary"]) #summary column for the 2nd record

Split lengths: [14732, 819, 818]
Features: ['id', 'dialogue', 'summary']

dialogue
Eric: MACHINE!
Rob: That's so gr8!
Eric: I know! And shows how Americans see Russian ;)
Rob: And it's really funny!
Eric: I know! I especially like the train part!
Rob: Hahaha! No one talks to the machine like that!
Eric: Is this his only stand-up?
Rob: Idk. I'll check.
Eric: Sure.
Rob: Turns out no! There are some of his stand-ups on youtube.
Eric: Gr8! I'll watch them now!
Rob: Me too!
Eric: MACHINE!
Rob: MACHINE!
Eric: TTYL?
Rob: Sure :)

summary
Eric and Rob are going to watch a stand-up on youtube.

dialogue
Lenny: Babe, can you help me with something?
Bob: Sure, what's up?
Lenny: Which one should I pick?
Bob: Send me photos
Lenny:  <file_photo>
Lenny:  <file_photo>
Lenny:  <file_photo>
Bob: I like the first ones best
Lenny: But I already have purple trousers. Does it make sense to have two pairs?
Bob: I have four black pairs :D :D
Lenny: yeah, but shouldn't I pick a different color?
Bob: what matte

We need to prepare the data for the seq2seq model:

### Preparing data for training for Sequence to sequence model

Lets say we have a row like this, a combination of a display.

{
  'dialogue' : "Hi! How are you?,
  
  'summary': "The speaker is asking how the other person is."
}

Then we would need the tonek Ids, which would convert the input words to tokens (we are using autotokenizers for that) which is needed for seqtoseq model, then attention mask to apply some cpecial characters to words in form of tokens to fine tune the se12se1 model. then we have labels which are tokens for the summary target.
So these are things we need to do for the train dataset before we pass it to the seq2seq model.

{
  'input_ids: [123,456,768....],

  'attention_mask': [1,1,1,1 ...],

  'lables': [2311, 356, 879, .....]
}

In [15]:
def convert_examples_to_features(example_batch):
  """
  tokenizer(...): Splits raw sentences into smaller units called tokens (words or sub-words) and maps them to unique integer IDs.
example_batch['dialogue']: The conversational text the model will read. It is the dialogue column of the Samsum data.
max_length = 1024: Limits the input to a maximum of 1,024 tokens.
truncation = True: Cuts off any text that exceeds the 1,024-token limit.
padding=True: Adds dummy tokens (zeros) to shorter sentences so every item in the batch has the exact same length.
return_tensors='pt': Outputs the data as PyTorch tensors.
"""
  input_encodings = tokenizer(example_batch['dialogue'], max_length = 1024, truncation = True, padding=True, return_tensors='pt')
  """
  What it does: Performs the exact same tokenization, truncation, and padding steps on the summary column.
  Why it matters: The model needs to see the correct mathematical representation of the target answer to learn how to generate it.
  """
  target_encodings = tokenizer(example_batch['summary'], max_length = 1024, truncation = True, padding=True, return_tensors='pt')

  """
  This is basically formatting the output
  input_ids: The sequence of numerical IDs representing the original dialogue words.
  attention_mask: A sequence of 1s and 0s telling the model which tokens are actual words (1) and which are just padding zeros (0).
  labels: The numerical IDs of the summary, which act as the ground-truth answer key during training.
  .tolist(): Converts the PyTorch tensors back into standard Python lists, which is the standard format required by Hugging Face dataset mapping functions.
  """
  return{
      'input_ids' : input_encodings['input_ids'].tolist(),
      'attention_mask' : input_encodings['attention_mask'].tolist(),
      'labels' : target_encodings['input_ids'].tolist()


  }

In [16]:
#Now the aforesaid function is applied to the whole dataset (dataset_samsum)
dataset_samsum_pt = dataset_samsum.map(convert_examples_to_features,batched = True)

Map:   0%|          | 0/14732 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

In [17]:
#You can seen there a other features too in the dataset
dataset_samsum_pt['train']
# Now ou will see three more columns have been added  'input_ids', 'attention_mask', 'labels'

Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 14732
})

Datacollatorforseq2seq is a special data collator designed for seq to seq models  like pegasus, T5 and BART) that helps in preparing batches for data mining.

In [18]:
from transformers import DataCollatorForSeq2Seq, TrainingArguments, Trainer
#from transformers.trainer_utils import EvaluationStrategy

seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

In [20]:


#Since we are limited by CPU. lets use num of train epochs as 1, later when we have GPU,
# we can increase it.
trainer_args = TrainingArguments(
    output_dir = 'pegasus-samsum', num_train_epochs=1, warmup_steps=500,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    weight_decay=0.1, logging_steps=10,
    #evaluation_strategy='steps',
    eval_steps=500, save_steps=1e6, gradient_accumulation_steps=16
)

In [21]:
#Now we execute our trainer> here we are giving the test dataset instead of the train so the
#training happens quickly.
trainer = Trainer(model=model_pegasus, args=trainer_args,
                  data_collator=seq2seq_data_collator,
                  train_dataset=dataset_samsum_pt["train"],
                  eval_dataset=dataset_samsum_pt["validation"]
                  )

In [22]:
trainer.train()

Step,Training Loss
10,150.012366
20,145.017761
30,147.678052
40,145.019250
50,140.873926
60,141.056519
70,138.757654
80,137.965112
90,134.036548
100,134.552075


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=921, training_loss=57.02656507647387, metrics={'train_runtime': 8049.6927, 'train_samples_per_second': 1.83, 'train_steps_per_second': 0.114, 'total_flos': 3.0452325061656576e+16, 'train_loss': 57.02656507647387, 'epoch': 1.0})

In [23]:
#It generates summaries for the test data, compares them against human-written summaries, and calculates ROUGE scores to measure text quality.

"""
What it does: Splits a large list of text data into smaller, manageable chunks (batches).
Why it matters: Generative models require high GPU memory; processing the entire dataset at once would cause an "out-of-memory" error.
The yield keyword makes it memory-efficient.

"""

def generate_batch_sized_chunks(list_of_elements, batch_size):
    """split the dataset into smaller batches that we can process simultaneously
    Yield successive batch-sized chunks from list_of_elements."""
    for i in range(0, len(list_of_elements), batch_size):
      yield list_of_elements[i : i + batch_size]

"""
Evaluation core setup
Splits both the input texts (column_text) and the ground-truth summaries (column_summary) into identical parallel batches.
"""
def calculate_metric_on_test_ds(dataset, metric, model, tokenizer,
                                batch_size=16, device = device, column_text= "article",
                                column_summary="highlights"):

  #whetever article we are giving, we are converting to a batch size.

    article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))
    """
    Processing and Tokenizing:
    tqdm: Displays a visual progress bar in the console.
    tokenizer(...): Converts the current batch of human conversations into numerical tensors.
    padding="max_length": Forces every sequence in the batch to be exactly 1024 tokens long by adding zeros.

    """

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches), total=len(article_batches)):

        inputs = tokenizer(article_batch, max_length=1024, truncation=True,
                        padding="max_length", return_tensors="pt")

        """
        Generating Summaries (Inference):
        .to(device): Moves the input tensors to your hardware accelerator (GPU or CPU).
        model.generate(...): Triggers the model to predict and output a new summary.
        num_beams=8: Uses Beam Search to explore 8 possible word sequences simultaneously, yielding higher quality text than picking one word at a time.
        length_penalty=0.8: A value under 1.0 encourages the model to generate shorter, more concise sentences.
        max_length=128: Caps the generated summary at 128 tokens.


        """

        summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                                   attention_mask=inputs["attention_mask"].to(device),
                                   length_penalty=0.8, num_beams=8, max_length=128)

        ''' parameter for length penalty ensures that the model does not generate sequences that are too long. '''

        #Finally, we decode the generated texts,
        #replace the token and add the decoded texts with the references to the metric.

        """
        Decoding and Metric Accumulation:
        tokenizer.decode(...): Converts the model's numerical outputs back into human-readable text.
        skip_special_tokens=True: Removes structural tokens like <pad>, <s>, or </s>.
        metric.add_batch(...): Stores the machine-generated summaries (predictions) and human summaries (references) in a temporary memory buffer.

        """

        decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True,
                                            clean_up_tokenization_spaces=True)
                              for s in summaries]

        decoded_summaries = [d.replace("", " ") for d in decoded_summaries]

        metric.add_batch(predictions=decoded_summaries, references=target_batch)

    #finally compute and return the ROGUE scores.
    score = metric.compute()
    return score



In [25]:
#Initializing the ROUGE Object
"""
The Metrics:
ROUGE-1: Measures overlap of single words (unigrams).
ROUGE-2: Measures overlap of two-word pairs (bigrams).
ROUGE-L: Measures the Longest Common Subsequence (sentence structure/word order).

"""
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_metric = evaluate.load('rouge')

"""
Execution and Formatting Results
dataset_samsum['test'][0:10]: Runs evaluation on just the first 10 samples of the test set to save time.
score[rn].mid.fmeasure: Extracts the exact statistical midpoint F1-score for each ROUGE type.
pd.DataFrame(...): Formats the final mathematical dictionary into a clean, readable Pandas table.
"""
score = calculate_metric_on_test_ds(
    dataset_samsum['test'][0:10], rouge_metric, trainer.model, tokenizer, batch_size = 2, column_text = 'dialogue', column_summary= 'summary')

rouge_dict = dict((rn, score[rn]) for rn in rouge_names) # Changed score[rn].mid.fmeasure to score[rn]
pd.DataFrame(rouge_dict, index = [f'pegasus'] )
#

100%|██████████| 5/5 [00:19<00:00,  3.96s/it]


,rouge1,rouge2,rougeL,rougeLsum
pegasus,0.021519,0.0,0.021484,0.021485


In [26]:
#Save the model
model_pegasus.save_pretrained("pegasus-samsum-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [27]:
#Save the tokenizer
tokenizer.save_pretrained("tokenizer")

('tokenizer/tokenizer_config.json', 'tokenizer/tokenizer.json')

In [28]:
# we can load the tokenizer as well

tokenizer = AutoTokenizer.from_pretrained("/content/tokenizer")


In [33]:
#prediction

gen_kwargs = {"length_penalty": 0.8, "num_beams":8, "max_length": 128}

sample_text = dataset_samsum["test"][0]["dialogue"]

reference = dataset_samsum["test"][0]["summary"]
#pipeline from huggingface

# Explicitly load the model first
from transformers import AutoModelForSeq2SeqLM
loaded_model = AutoModelForSeq2SeqLM.from_pretrained("/content/pegasus-samsum-model").to(device)

# Then create the pipeline with the loaded model object and tokenizer
# Bypassing the pipeline due to "Unknown task summarization" error
# pipe = pipeline("summarization", model=loaded_model, tokenizer=tokenizer)

## Direct generation for prediction
inputs = tokenizer(sample_text, max_length=1024, truncation=True, return_tensors="pt")
summaries = loaded_model.generate(input_ids=inputs["input_ids"].to(device),
                                 attention_mask=inputs["attention_mask"].to(device),
                                 **gen_kwargs)
model_summary = tokenizer.decode(summaries[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)


print("Dialogue:")
print(sample_text)


print("\nReference Summary:")
print(reference)

print("\nModel Summary:")
#here actually executing the pipeline, giving the sample text and passing the arguements
print(model_summary)

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

Dialogue:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Reference Summary:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

Model Summary:
Amanda can't find Betty's number. Larry called Betty the last time they were at the park together. Hannah wants Amanda to text him instead.
